# Audio Tabula Rasa — Toy Case: Consonance Discovery

Can a generator discover musical intervals from physics alone, with **zero training on music**?

**Setup:**
- A tiny MLP outputs two frequencies (an interval).
- A reward model based on the Sethares (1993) dissonance curve scores them. This is grounded in psychoacoustics — critical band theory of the basilar membrane — not in any music corpus.
- REINFORCE policy gradient.

**Predicted result:** the generator discovers consonant intervals (octaves, fifths, fourths, sixths) without being told they exist.

In [ ]:
!pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import Counter

## 1. Reward model: Sethares dissonance

Roughness between two pure partials, summed over the first 6 harmonics of each complex tone. Consonance arises when harmonics coincide rather than collide — pure physics.

In [ ]:
def _sethares_pair(f1, f2, a1=1.0, a2=1.0):
    fmin = min(f1, f2)
    fdif = abs(f1 - f2)
    if fdif < 1e-9:
        return 0.0
    s = 0.24 / (0.0207 * fmin + 18.96)
    return a1 * a2 * (np.exp(-3.5 * s * fdif) - np.exp(-5.75 * s * fdif))

def total_dissonance(f1, f2, n_harmonics=6):
    diss = 0.0
    for i in range(1, n_harmonics + 1):
        for j in range(1, n_harmonics + 1):
            diss += _sethares_pair(i * f1, j * f2, 1.0 / i, 1.0 / j)
    return diss

def consonance_reward(f1, f2):
    return -total_dissonance(f1, f2)

def ratio_label(f1, f2, tol=0.02):
    r = max(f1, f2) / min(f1, f2)
    named = {1.0:'unison',1.067:'m2',1.125:'M2',1.2:'m3',1.25:'M3',
             1.333:'P4',1.414:'tritone',1.5:'P5',1.6:'m6',1.667:'M6',
             1.8:'m7',1.875:'M7',2.0:'octave'}
    best = min(named, key=lambda k: abs(k - r))
    if abs(best - r) / best < tol:
        return named[best]
    return f'non-musical'

**Sanity check — visualize the reward landscape:**

In [ ]:
base = 220.0
ratios = np.linspace(1.0, 2.05, 400)
diss = [-consonance_reward(base, base*r) for r in ratios]
plt.figure(figsize=(10, 4))
plt.plot(ratios, diss, lw=2, color='crimson')
for r, name in [(1.0,'1:1'),(1.25,'5:4'),(1.333,'4:3'),(1.5,'3:2'),(1.667,'5:3'),(2.0,'2:1')]:
    plt.axvline(r, color='gray', ls=':', alpha=0.6)
    plt.text(r, max(diss)*1.02, name, ha='center', fontsize=9)
plt.xlabel('Frequency ratio'); plt.ylabel('Dissonance')
plt.title('Sethares dissonance — consonant intervals appear as minima from PHYSICS, not training'); plt.grid(alpha=0.3); plt.show()

## 2. Generator: tabula-rasa policy

In [ ]:
class ToyIntervalGenerator(nn.Module):
    F_MIN, F_MAX = 110.0, 880.0
    def __init__(self, latent_dim=8, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 4),
        )
        self.latent_dim = latent_dim
    def forward(self, z):
        p = self.net(z)
        mean = torch.sigmoid(p[:, :2]) * (self.F_MAX - self.F_MIN) + self.F_MIN
        std = torch.exp(p[:, 2:].clamp(-2.0, 3.0))
        return mean, std
    def sample(self, n):
        z = torch.randn(n, self.latent_dim)
        mean, std = self.forward(z)
        dist = torch.distributions.Normal(mean, std)
        f = dist.rsample().clamp(self.F_MIN, self.F_MAX)
        return f, dist.log_prob(f).sum(-1)

## 3. Train via REINFORCE

In [ ]:
torch.manual_seed(0); np.random.seed(0)
gen = ToyIntervalGenerator()
opt = torch.optim.Adam(gen.parameters(), lr=1e-3)
history = []

for step in range(1500):
    freqs, log_prob = gen.sample(64)
    rewards = torch.tensor([consonance_reward(float(f[0]), float(f[1]))
                            for f in freqs.detach().numpy()], dtype=torch.float32)
    adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    loss = -(log_prob * adv).mean()
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == 1499:
        with torch.no_grad():
            ef, _ = gen.sample(256)
            er = [consonance_reward(float(f[0]), float(f[1])) for f in ef.numpy()]
            labels = [ratio_label(float(f[0]), float(f[1])) for f in ef.numpy()]
        history.append({'step': step, 'reward': float(np.mean(er)),
                        'top': Counter(labels).most_common(3)})
        print(f"[{step:5d}] reward={history[-1]['reward']:+.3f}  top={history[-1]['top']}")

## 4. Final evaluation — what did it discover?

In [ ]:
with torch.no_grad():
    ef, _ = gen.sample(1000)
    ratios = (ef.max(1).values / ef.min(1).values).numpy()
    labels = [ratio_label(float(f[0]), float(f[1])) for f in ef.numpy()]

plt.figure(figsize=(12, 4))
plt.hist(ratios, bins=80, color='steelblue', alpha=0.8)
for r, name in [(1.25,'M3'),(1.333,'P4'),(1.5,'P5'),(1.667,'M6'),(2.0,'octave')]:
    plt.axvline(r, color='red', ls='--', alpha=0.5)
    plt.text(r, plt.ylim()[1]*0.95, name, ha='center', fontsize=9, color='red')
plt.xlabel('Frequency ratio'); plt.ylabel('Count')
plt.title('Distribution of intervals after training'); plt.show()
print('Top intervals:', Counter(labels).most_common(8))